In [32]:
import pandas as pd
import torch
import torch.nn
import torch.optim as optim
from sklearn.model_selection import  train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import  LabelEncoder
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
df = pd.read_csv(r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [5]:
# Preprocessing
import re
df["review"] = df["review"].str.lower()

In [6]:
# Remove URLs

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text
df["review"] = df["review"].apply(remove_urls)

In [7]:
# Remove Punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
df["review"] = df["review"].apply(remove_punctuations)

In [8]:
# Remove HTML tags

def remove_html(text):
    text = re.sub(r"[<.*?>]", "", text)
    return text
df["review"] = df["review"].apply(remove_html)

In [ ]:
# Remove Stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        text = text.replace(word, "")
    return text
df["review"] = df["review"].apply(remove_stopwords)

In [ ]:
# Stemming

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
# Encoding & Vectorization

le = LabelEncoder()
y = df["sentiment"] = le.fit_transform(df["sentiment"])
df["sentiment"].value_counts()

sentiment
1    24884
0    24698
Name: count, dtype: int64

In [15]:
# TF-IDF Vectorization

tf = TfidfVectorizer(max_features=5000)
x = tf.fit_transform(df["review"])

In [18]:
# Train Test Split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train.shape, x_test.shape

((39665, 5000), (9917, 5000))

In [19]:
# Convert TensorDataset
x_train = x_train.toarray()
x_test  = x_test.toarray()

In [24]:
# Convert TensorDataset

train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test).float()
)

In [28]:
#  Dataloader 

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_set     = DataLoader(test_set, batch_size=64, shuffle=True)

In [41]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [44]:
input_size = x_train.shape[1]

model = RNN(input_size)

criteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters())